# Type II excitability with the resonant band-pass selector

Companion code for *Excitability Types and Bursting with a Two-Block Spiking Primitive*,
Section V. **This notebook produces Fig. 4.**

The selector is the second-order resonant band-pass

$$H_\mathrm{r}(s) = \frac{(\omega_n/Q)\,s}{s^2 + (\omega_n/Q)\,s + \omega_n^2},
\qquad \omega_n = 2\pi f_c,$$

in controllable canonical form, so the output equation is explicit. No algebraic loop, no
regularizing low-pass, no stiffness, and `Tsit5` is enough. The model is parametrized by
the center frequency in hertz, so with $f_c = 1$ Hz every rate-input curve starts at 1 Hz.

Three claims of the paper are checked here: the criticality condition
$\mathrm{sig}_k'''(u_\mathrm{th}) = 2k(2k-3)$, the bistability window that the subcritical
onset opens, and the rate-input curves of both selectors.

In [ ]:
using Plots, LaTeXStrings, DifferentialEquations
using Printf, Statistics, ProgressMeter, Plots.PlotMeasures

gr(guidefontsize = 14, tickfontsize = 12, legendfontsize = 12, margin = 5Plots.mm, grid = true)
myApple  = RGBA(187/255, 206/255, 131/255, 1)
myBlue   = RGBA(131/255, 174/255, 218/255, 1)
myOrange = RGBA(241/255, 175/255, 113/255, 1)
myPurple = RGBA(169/255,  90/255, 179/255, 1)
myGray   = RGBA(150/255, 150/255, 150/255, 1)
default(fmt = :png);

## The model

In [ ]:
Base.@kwdef struct BPNeuron
    k::Float64   = 3.0        # sigmoid gain, k > 1 to fire, k > 3/2 for a hard onset
    Q::Float64   = 1.0        # quality factor, Q > 1/2 for conjugate poles
    fc::Float64  = 1.0        # center frequency [Hz]
    u::Function  = t -> 0.0   # external input, at the sigmoid port
end

const_input(a) = t -> a

omega(p::BPNeuron) = 2pi * p.fc

# algebraic signals from the states X = [x1, x2]
bp_out(x2, p)     = (omega(p) / p.Q) * x2                # w
sig_in(x2, p, t)  = bp_out(x2, p) + p.u(t)               # v = u + w
sig_out(x2, p, t) = tanh(p.k * sig_in(x2, p, t))         # y

function rhs!(dX, X, p::BPNeuron, t)
    x1, x2 = X
    y = sig_out(x2, p, t)
    dX[1] = x2
    dX[2] = -omega(p)^2 * x1 - (omega(p) / p.Q) * x2 + y
    return nothing
end

# resting point of the loop: x2 = 0 and x1 = tanh(k*u)/wn^2
equilibrium(p::BPNeuron, u::Float64) = [tanh(p.k * u) / omega(p)^2, 0.0]

# start off the resting point with a prescribed initial feedback signal w0, which is the
# scale-free signal of the loop and therefore means the same thing at every fc and Q
kickstart(p::BPNeuron, u, w0) = equilibrium(p, u) .+ [0.0, w0 * p.Q / omega(p)]

function simulate(p::BPNeuron; tspan = (0.0, 10.0), X0 = nothing,
                  abstol = 1e-10, reltol = 1e-9, dtmax = 0.005, tstops = Float64[])
    X0 === nothing && (X0 = equilibrium(p, p.u(tspan[1])))
    solve(ODEProblem(rhs!, X0, tspan, p), Tsit5(); abstol = abstol, reltol = reltol,
          dtmax = dtmax, tstops = tstops)
end

function signals(sol, p::BPNeuron)
    t  = sol.t
    x2 = [s[2] for s in sol.u]
    u  = p.u.(t)
    w  = (omega(p) / p.Q) .* x2
    return (t = t, x1 = [s[1] for s in sol.u], x2 = x2, u = u, w = w,
            v = u .+ w, y = tanh.(p.k .* (u .+ w)))
end

function plot_output(sol, p::BPNeuron; label = "")
    s = signals(sol, p)
    plot(s.t, s.y; lw = 2, c = myPurple, xlabel = "time [s]", ylabel = L"y",
         ylims = (-1.05, 1.05), legend = false, size = (900, 280),
         title = label, titlefontsize = 11)
end

## Threshold and criticality

The rheobase belongs to the nonlinearity, $|u_\mathrm{th}| = \mathrm{arctanh}\sqrt{1-1/k}/k$,
and the sign of the third derivative of the sigmoid at that threshold decides whether the
onset is soft or hard. Both are closed form, so no simulation is needed here.

In [ ]:
sigslope(p::BPNeuron, u) = p.k * sech(p.k * u)^2      # a(u) = sig_k'(u)
rheobase(k)              = atanh(sqrt(1 - 1/k)) / k   # |u_th|
criticality(k)           = 2k * (2k - 3)              # sig_k'''(u_th) for tanh

@printf("%6s %10s %16s   %s\n", "k", "u_th", "sig'''(u_th)", "onset")
println("-"^58)
for k in [1.1, 1.2, 1.5, 2.0, 3.0, 5.0, 8.0]
    c = criticality(k)
    @printf("%6.2f %10.4f %16.3f   %s\n", k, rheobase(k), c,
            c > 0 ? "subcritical, hard" : c < 0 ? "supercritical, soft" : "degenerate")
end

## The bistability window

For $k > 3/2$ the onset is subcritical, so the cycle survives below the Hopf down to a fold
of limit cycles and rest coexists with spiking in between. At $k = 3$, $Q = 1$ that window
runs from $u_\mathrm{th} = 0.382$ to roughly $0.497$. Same parameters, same input, two
initial conditions, two attractors. This coexistence is the budget the slow loop of
`simulate_neuron_elliptic.ipynb` spends.

In [ ]:
uu = 0.45      # inside the window
p  = BPNeuron(k = 3.0, Q = 1.0, fc = 1.0, u = const_input(uu))
for w0 in [0.3, 0.5]
    sol = simulate(p; tspan = (0.0, 20.0), X0 = kickstart(p, uu, w0))
    display(plot_output(sol, p; label = "u = $uu, initial w = $w0"))
end

## Rate-input curves

One independent simulation per value of $u$, with $u$ held constant throughout that run.
Two initializations answer two different questions:

* **kicked**, far off the equilibrium, so the run lands on the large cycle whenever one
  exists and the sweep terminates at the fold of limit cycles, past the Hopf;
* **rest-initialized**, with only a numerical seed, so inside the bistable window the seed
  stays within the basin of rest and the sweep terminates at the Hopf itself.

The gap between the two endpoints is the bistability window. Two guards are needed in the
frequency measurement, otherwise the sweep lies: a decay test comparing the amplitude on
two late windows, since close to the Hopf the transient decays very slowly and looks
periodic over any finite horizon, and a regularity test on the measured periods.

The seed cannot be exactly zero, since the equilibrium is an exact fixed point of the
vector field. It must also stay small, since a seed larger than the unstable cycle would
cross the basin boundary and land back on the fold. The growth rate of the seed is
$\sigma = \omega_n(a-1)/2Q$, which vanishes at threshold, so the horizon is scaled as
$1/\sigma$ rather than held fixed, and the input grid is refined just below
$u_\mathrm{th}$. Neither move changes the endpoint, they only set how closely it is
resolved.

In [ ]:
"""
    limit_cycle(p; T, kick)

Frequency in hertz and output amplitude of the attracting limit cycle, for a constant
input. Returns f = 0 when the cell settles to rest.
"""
function limit_cycle(p::BPNeuron; T = 100.0, kick = 2.0, dt = 0.002, discard = 0.6)
    u0  = p.u(0.0)
    sol = solve(ODEProblem(rhs!, kickstart(p, u0, kick), (0.0, T), p), Tsit5();
                abstol = 1e-10, reltol = 1e-9, saveat = dt)
    t = sol.t
    w = (omega(p) / p.Q) .* [s[2] for s in sol.u]   # feedback signal, scale free

    # (1) decay test: is the oscillation steady, or dying out?
    i1 = findfirst(x -> x >= 0.45 * T, t)
    i2 = findfirst(x -> x >= 0.60 * T, t)
    i3 = findfirst(x -> x >= 0.85 * T, t)
    A1 = maximum(w[i1:i2]) - minimum(w[i1:i2])
    A2 = maximum(w[i3:end]) - minimum(w[i3:end])
    (A2 < 0.02 || A2 < 0.98 * A1) && return (f = 0.0, amp = 0.0)

    # (2) upward zero crossings of w, linearly interpolated
    i0    = findfirst(x -> x >= discard * T, t)
    cross = Float64[]
    for i in i0:(length(t) - 1)
        if w[i] < 0 && w[i+1] >= 0
            push!(cross, t[i] - w[i] * (t[i+1] - t[i]) / (w[i+1] - w[i]))
        end
    end
    length(cross) < 3 && return (f = 0.0, amp = 0.0)

    per = diff(cross)
    (std(per) > 0.03 * mean(per)) && return (f = 0.0, amp = 0.0)

    y = tanh.(p.k .* (p.u.(t[i0:end]) .+ w[i0:end]))
    return (f = 1 / mean(per), amp = 0.5 * (maximum(y) - minimum(y)))
end

"""Kicked sweep over a grid of constant inputs. Ends at the fold of cycles."""
function fI_curve(; k = 3.0, Q = 1.0, fc = 1.0, us = range(0.0, 0.6; length = 91),
                    kick = 2.0, T = 100.0)
    f = zeros(length(us)); a = zeros(length(us))
    @showprogress for (i, u) in enumerate(us)
        r = limit_cycle(BPNeuron(k = k, Q = Q, fc = fc, u = const_input(u)); T = T, kick = kick)
        f[i] = r.f; a[i] = r.amp
    end
    return collect(us), f, a
end

"""Horizon long enough for a tiny seed to reach the cycle at input u."""
function horizon(p::BPNeuron, u; nefolds = 9.0, Tmin = 120.0, Tmax = 800.0)
    sigma = omega(p) * (sigslope(p, u) - 1) / (2 * p.Q)
    sigma <= 0 && return Tmin
    return clamp(2.5 * nefolds / sigma, Tmin, Tmax)
end

"""Coarse sweep plus a fine band just below the rheobase of this k."""
function refined_grid(k; umax = 0.6, ncoarse = 91, nfine = 25, width = 0.02)
    uH     = rheobase(k)
    coarse = collect(range(0.0, umax; length = ncoarse))
    fine   = collect(range(max(uH - width, 0.0), uH; length = nfine))
    return sort(unique(vcat(coarse, fine)))
end

"""Rest-initialized sweep. Ends at the rheobase."""
function fI_curve_rest(; k = 3.0, Q = 1.0, fc = 1.0, us = range(0.0, 0.6; length = 91),
                         seed = 1e-3, Tmax = 800.0)
    f = zeros(length(us)); a = zeros(length(us))
    @showprogress for (i, u) in enumerate(us)
        p = BPNeuron(k = k, Q = Q, fc = fc, u = const_input(u))
        r = limit_cycle(p; T = horizon(p, u; Tmax = Tmax), kick = seed, dt = 0.005)
        f[i] = r.f; a[i] = r.amp
    end
    return collect(us), f, a
end

The top panel of Fig. 4 is the closed-form rate of the high-pass selector with the
saturation sigmoid. It is parametrized by the gap $g = 1 - k|u|$ rather than by $u$, so the
band edge is reachable to machine precision with no cancellation.

In [ ]:
# closed-form rate for H_hp with the saturation sigmoid, tau = 1
fgap(g, k, tau) = 1 / (tau*log((2k - 2 + g)/g) + tau*log((2k - g)/(2 - g)))

# tick helper, shared with the other figure notebooks
fmtnum(x, d) = Printf.format(Printf.Format("%.$(d)f"), iszero(x) ? 0.0 : x)

function niceticks(lo, hi; n = 4)
    raw  = (hi - lo) / n
    mag  = 10.0^floor(log10(raw))
    r    = raw / mag
    step = (r < 1.5 ? 1.0 : r < 3.0 ? 2.0 : r < 7.0 ? 5.0 : 10.0) * mag
    v    = collect(ceil(lo/step - 1e-9)*step : step : floor(hi/step + 1e-9)*step)
    d    = max(0, Int(-floor(log10(step))))
    (v, [latexstring(fmtnum(x, d)) for x in v])
end

# both selectors are even in u, so half a sweep is enough
function mirror(u, f)
    isempty(u) && return (u, f)
    i = u[1] > 1e-12 ? 1 : 2
    (vcat(-reverse(u[i:end]), u), vcat(reverse(f[i:end]), f))
end

The sweep below runs both initializations for four sigmoid gains. It takes a few minutes.

In [ ]:
ks_fig4  = [2.0, 3.0, 5.0, 8.0]
Q_fig4   = 1.0
fI_cache = Dict{Float64, NamedTuple}()

@printf("%6s %10s %10s %10s %10s\n", "k", "u_th", "u_end", "fold", "window")
println("-"^52)
for k in ks_fig4
    grd = refined_grid(k)
    Tk  = k < 3.0 ? 1500.0 : 800.0
    usr, fr, _ = fI_curve_rest(k = k, Q = Q_fig4, us = grd, Tmax = Tk)
    usk, fk, _ = fI_curve(k = k, Q = Q_fig4, us = grd, kick = 2.0)
    mr = fr .> 0
    mk = (fk .> 0) .& (usk .> rheobase(k))
    fI_cache[k] = (uH = rheobase(k), u_rest = usr[mr], f_rest = fr[mr],
                   u_kick = usk[mk], f_kick = fk[mk])
    uend  = any(mr) ? usr[mr][end] : NaN
    ufold = any(mk) ? usk[mk][end] : NaN
    @printf("%6.1f %10.4f %10.4f %10.4f %10.4f\n",
            k, rheobase(k), uend, ufold, isnan(ufold) ? 0.0 : ufold - rheobase(k))
end

## Fig. 4

In [ ]:
# --- Fig. 4: rate-input curves for the two selectors ------------------------
cols_fig4 = [myBlue, myApple, myPurple, myOrange]
lss_fig4  = [:solid, :dash, :dot, :dashdot]
FSg, FSt, FSl, FSa = 14, 12, 11, 13
XLIM = (-0.56, 0.56)
xt   = niceticks(-0.5, 0.5)

# linear over the band, log over the last three decades of the descent
gg = sort(unique(vcat(collect(range(1.0, 1e-3; length = 800)),
                      10 .^ range(-3.0, -280.0; length = 400))); rev = true)

# --- top panel: H_hp
pT = plot(; xlims = XLIM, ylims = (0.0, 0.58),
          ylabel = L"f\,\tau", xlabel = "",
          xticks = (xt[1], fill("", length(xt[1]))),
          yticks = niceticks(0.0, 0.5),
          legend = :top, legend_columns = 4, legendfontsize = FSl,
          guidefontsize = FSg, tickfontsize = FSt,
          foreground_color_legend = nothing, background_color_legend = nothing)
for (k, c, s) in zip(ks_fig4, cols_fig4, lss_fig4)
    uu = (1 .- gg) ./ k
    ff = [fgap(g, k, 1.0) for g in gg]
    plot!(pT, vcat(-reverse(uu), uu), vcat(reverse(ff), ff);
          lw = 3.0, lc = c, ls = s, label = "")
    plot!(pT, [NaN], [NaN]; lw = 1.5, lc = c, ls = s,
          label = latexstring("k = $(Int(k)) \\quad"))
    scatter!(pT, [-1/k, 1/k], [0.0, 0.0]; ms = 6.0, mc = c, msc = c, label = "")
end
annotate!(pT, -0.53, 0.575, text(L"H_\mathrm{hp}", FSa, :left, :top, :black))

# --- bottom panel: H_r, curves first, markers on top of them
pB = plot(; xlims = XLIM, ylims = (0.0, 1.15),
          xlabel = L"u", ylabel = L"f/f_c",
          xticks = xt, yticks = niceticks(0.0, 1.0),
          legend = :bottomleft, legendfontsize = FSl,
          guidefontsize = FSg, tickfontsize = FSt,
          foreground_color_legend = nothing, background_color_legend = nothing)
for (k, c, s) in zip(ks_fig4, cols_fig4, lss_fig4)
    r = fI_cache[k]
    ur, fr = mirror(r.u_rest, r.f_rest)
    isempty(ur) || plot!(pB, ur, fr; lw = 3.0, lc = c, ls = s, label = "")
    if !isempty(r.u_kick)
        plot!(pB,  r.u_kick, r.f_kick; lw = 3.0, lc = c, ls = s, linealpha = 0.35, label = "")
        plot!(pB, -r.u_kick, r.f_kick; lw = 3.0, lc = c, ls = s, linealpha = 0.35, label = "")
    end
end
for (k, c) in zip(ks_fig4, cols_fig4)
    r = fI_cache[k]
    isempty(r.u_rest) || scatter!(pB, [-r.u_rest[end], r.u_rest[end]],
        fill(r.f_rest[end], 2); ms = 6.0, mc = c, msc = c, label = "")
    isempty(r.u_kick) || scatter!(pB, [-r.u_kick[end], r.u_kick[end]],
        fill(r.f_kick[end], 2); ms = 6.0, mc = :white, msc = c, msw = 1.8, label = "")
end
# marker key, plotted outside the limits so that only the legend entries show
scatter!(pB, [9.0], [9.0]; ms = 6.0, mc = :black, msc = :black,
         label = L"\mathrm{onset\ from\ rest}")
scatter!(pB, [9.0], [9.0]; ms = 6.0, mc = :white, msc = :black, msw = 1.8,
         label = L"\mathrm{fold\ of\ cycles}")
hline!(pB, [1.0]; ls = :dot, lc = myGray, lw = 1, label = "")
annotate!(pB, -0.53, 1.15, text(L"H_\mathrm{r}", FSa, :left, :top, :black))

fig4 = plot(pT, pB; layout = (2, 1), link = :x, size = (600, 500),
            margins = 0Plots.mm, left_margin = 0Plots.mm,
            bottom_margin = 0Plots.mm, right_margin = 2Plots.mm, top_margin = 1Plots.mm)

mkpath("figures")
savefig(fig4, "figures/fig4.pdf")
fig4